# UV Training Set Generation

> Notebook purpose: generate synthetic UV-grid datasets, apply cleaning, and export reproducible JSON files for model training.

## Workflow
1. Generate base structures with mixed geometry types.
2. Apply optional point perturbations and cleaning.
3. Visualize sample structures.
4. Run parameter sweeps and export datasets.

In [2]:
import json, random, os
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import numpy as np
from typing import List, Dict, Tuple, Optional
import math

from Training_sett_scripts import build_structure_with_arc_curvature, build_structure_with_skew, build_structure_with_rotation_exCol, add_shifted_points_suffix, shift_random_points, plot_structure, build_structure_with_rotation_exCol_removePart


In [ ]:
# Generate training set (configuration A)
N_rotated = 1000

rng = random.Random(42)
structures_with_rotation = []

for i in range(N_rotated):
    # Balanced sampling across structure families
    r_float = rng.random()

    if r_float < 0.25:
        s = build_structure_with_arc_curvature(
            rng,
            rows_range=(3, 15),
            cols_range=(3, 15),
            row_gap_domain_mm=(4000, 6000),
            col_gap_domain_mm=(4000, 6000),
            rotation_domain_deg=(-45, 45),
            arc_curvature_domain=(-0.00001, 0.00001),
            random_columns_range=(0, 2),
            remove_points_range=(0, 2),
        )
    elif r_float < 0.50:
        s = build_structure_with_skew(
            rng,
            rows_range=(3, 15),
            cols_range=(3, 15),
            row_gap_domain_mm=(4000, 8000),
            col_gap_domain_mm=(4000, 8000),
            rotation_domain_deg=(-45, 45),
            skew_domain_deg=(-15, 15),
            random_columns_range=(0, 2),
            remove_points_range=(0, 2),
        )
    elif r_float < 0.75:
        s = build_structure_with_rotation_exCol(
            rng,
            rows_range=(3, 15),
            cols_range=(3, 15),
            row_gap_domain_mm=(4000, 8000),
            col_gap_domain_mm=(4000, 8000),
            rotation_domain_deg=(-45, 45),
            random_columns_range=(0, 2),
            remove_points_range=(0, 1),
        )
    else:
        s = build_structure_with_rotation_exCol_removePart(
            rng,
            rows_range=(3, 15),
            cols_range=(3, 15),
            row_gap_domain_mm=(4000, 8000),
            col_gap_domain_mm=(4000, 8000),
            rotation_domain_deg=(-45, 45),
            random_columns_range=(0, 2),
            remove_points_range=(0, 0),
            removal_probability=0.7,
        )

    structures_with_rotation.append(s)

# Save generated dataset
out_file_rotated = "JSON/rotated_combination_arc_skew_ort_3-15_4-8_n1.json"
with open(out_file_rotated, "w") as f:
    json.dump({"structures": structures_with_rotation}, f, indent=2)

# Dataset summary
total_random_cols = sum(s.get("num_random_columns", 0) for s in structures_with_rotation)
total_grid_points = sum(s["R"] * s["C"] for s in structures_with_rotation)
avg_random_cols = total_random_cols / len(structures_with_rotation) if structures_with_rotation else 0
min_random_cols = min(s.get("num_random_columns", 0) for s in structures_with_rotation)
max_random_cols = max(s.get("num_random_columns", 0) for s in structures_with_rotation)
avg_percentage = (total_random_cols / total_grid_points * 100) if total_grid_points > 0 else 0

print(f"Saved {N_rotated} structures to {os.path.abspath(out_file_rotated)}")
print("Rotation range: -45 to 45 degrees")
print("Random columns per structure:")
print(f"  Average: {avg_random_cols:.1f}")
print(f"  Min/Max: {min_random_cols}/{max_random_cols}")
print(f"  Total random columns: {total_random_cols}")
print(f"  Avg random-column ratio: {avg_percentage:.2f}%")

# Point-shift augmentation settings
in_path = out_file_rotated
out_path = add_shifted_points_suffix(in_path)
seed = 42
point_count_domain = (1, 5)
dx_domain_mm = (-50.0, 50.0)
dy_domain_mm = (-50.0, 50.0)

rng = random.Random(seed)

with open(in_path, "r") as f:
    data = json.load(f)

if "structures" not in data or not isinstance(data["structures"], list):
    raise ValueError("Input JSON must contain top-level 'structures': [...]")

for s in data["structures"]:
    shift_random_points(s, rng, point_count_domain, dx_domain_mm, dy_domain_mm)

with open(out_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved augmented file to:\n{os.path.abspath(out_path)}")

# Visualize a subset
in_file = out_path
with open(in_file, "r") as f:
    data = json.load(f)

max_n = 1000
structures = data["structures"][:max_n]
print(f"Loaded {len(structures)} structures")

max_n = min(max_n, len(structures))
limited_structures = structures[:max_n]

widgets.interact(
    plot_structure,
    idx=widgets.IntSlider(min=0, max=max_n - 1, step=1, value=0),
    structures=widgets.fixed(structures),
)

Saved 1000 buildings with rotation to c:\Users\villemev\Documents\VS Projects\DGCNN_Grid_and_Substructure_Classification\JSON\rotated_combination_arc_skew_ort_3-15_4-8_n1.json
Rotation range: -45° to +45°
Random columns per structure:
  Average: 0.9 columns
  Min: 0, Max: 2
  Total random columns: 941
  Average percentage: 1.21% of grid points (max 10% allowed)
Saved updated file (all structures, points kept) to:
c:\Users\villemev\Documents\VS Projects\DGCNN_Grid_and_Substructure_Classification\JSON\rotated_combination_arc_skew_ort_3-15_4-8_n1_shifted_points.json
Loaded 1000 buildings


interactive(children=(IntSlider(value=0, description='idx', max=999), Output()), _dom_classes=('widget-interac…

<function Training_sett_scripts.plot_structure(idx, structures)>

In [ ]:
# Generate training set (configuration B)
N_rotated = 1000

rng = random.Random(42)
structures_with_rotation = []

for i in range(N_rotated):
    r_float = rng.random()

    if r_float < 0.25:
        s = build_structure_with_arc_curvature(
            rng,
            rows_range=(3, 15),
            cols_range=(3, 15),
            row_gap_domain_mm=(4000, 6000),
            col_gap_domain_mm=(4000, 6000),
            rotation_domain_deg=(-45, 45),
            arc_curvature_domain=(-0.00001, 0.00001),
            random_columns_range=(0, 15),
            remove_points_range=(0, 5),
        )
    elif r_float < 0.50:
        s = build_structure_with_skew(
            rng,
            rows_range=(3, 15),
            cols_range=(3, 15),
            row_gap_domain_mm=(4000, 8000),
            col_gap_domain_mm=(4000, 8000),
            rotation_domain_deg=(-45, 45),
            skew_domain_deg=(-15, 15),
            random_columns_range=(0, 15),
            remove_points_range=(0, 5),
        )
    elif r_float < 0.60:
        s = build_structure_with_rotation_exCol(
            rng,
            rows_range=(3, 15),
            cols_range=(3, 15),
            row_gap_domain_mm=(4000, 12000),
            col_gap_domain_mm=(4000, 12000),
            rotation_domain_deg=(-45, 45),
            random_columns_range=(0, 15),
            remove_points_range=(0, 5),
            random_axis_range=(0, 6),
        )
    else:
        s = build_structure_with_rotation_exCol_removePart(
            rng,
            rows_range=(10, 20),
            cols_range=(10, 20),
            row_gap_domain_mm=(4000, 8000),
            col_gap_domain_mm=(4000, 8000),
            rotation_domain_deg=(-45, 45),
            random_columns_range=(0, 20),
            remove_points_range=(0, 3),
            removal_probability=1,
            max_removals=4,
            random_axis_range=(10, 20),
        )

    structures_with_rotation.append(s)

out_file_rotated = "JSON/rotated_combination_arc_skew_ort_3-15_12-20_n1_more_random2_all_rn42.json"
with open(out_file_rotated, "w") as f:
    json.dump({"structures": structures_with_rotation}, f, indent=2)

total_random_cols = sum(s.get("num_random_columns", 0) for s in structures_with_rotation)
total_grid_points = sum(s["R"] * s["C"] for s in structures_with_rotation)
total_removed = sum(s.get("num_removed_points", 0) for s in structures_with_rotation)
total_removal_areas = sum(s.get("num_removal_areas", 0) for s in structures_with_rotation)
multi_removal_count = sum(1 for s in structures_with_rotation if s.get("num_removal_areas", 0) > 1)

avg_random_cols = total_random_cols / len(structures_with_rotation) if structures_with_rotation else 0
min_random_cols = min(s.get("num_random_columns", 0) for s in structures_with_rotation)
max_random_cols = max(s.get("num_random_columns", 0) for s in structures_with_rotation)
avg_percentage = (total_random_cols / total_grid_points * 100) if total_grid_points > 0 else 0

print(f"Saved {N_rotated} structures to {os.path.abspath(out_file_rotated)}")
print("Rotation range: -45 to 45 degrees")
print("\nRandom columns per structure:")
print(f"  Average: {avg_random_cols:.1f}")
print(f"  Min/Max: {min_random_cols}/{max_random_cols}")
print(f"  Total random columns: {total_random_cols}")
print(f"  Avg random-column ratio: {avg_percentage:.2f}%")
print("\nRemoval statistics:")
print(f"  Total removed points: {total_removed}")
print(f"  Total removal areas: {total_removal_areas}")
print(f"  Structures with multiple removals: {multi_removal_count} ({multi_removal_count/N_rotated*100:.1f}%)")

in_path = out_file_rotated
out_path = add_shifted_points_suffix(in_path)
seed = 42
point_count_domain = (1, 5)
dx_domain_mm = (-50.0, 50.0)
dy_domain_mm = (-50.0, 50.0)

rng = random.Random(seed)

with open(in_path, "r") as f:
    data = json.load(f)

if "structures" not in data or not isinstance(data["structures"], list):
    raise ValueError("Input JSON must contain top-level 'structures': [...]")

for s in data["structures"]:
    shift_random_points(s, rng, point_count_domain, dx_domain_mm, dy_domain_mm)

with open(out_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved augmented file to:\n{os.path.abspath(out_path)}")

in_file = out_path
with open(in_file, "r") as f:
    data = json.load(f)

max_n = 1000
structures = data["structures"][:max_n]
print(f"Loaded {len(structures)} structures")

max_n = min(max_n, len(structures))
limited_structures = structures[:max_n]

widgets.interact(
    plot_structure,
    idx=widgets.IntSlider(min=0, max=max_n - 1, step=1, value=0),
    structures=widgets.fixed(structures),
)

Saved 1000 buildings with rotation to c:\Users\villemev\Documents\VS Projects\DGCNN_Grid_and_Substructure_Classification\JSON\rotated_combination_arc_skew_ort_3-15_12-20_n1_more_random2_all_rn42.json
Rotation range: -45° to +45°

Random columns per structure:
  Average: 7.0 columns
  Min: 0, Max: 20
  Total random columns: 7027
  Average percentage: 5.01% of grid points

Removal statistics:
  Total removed points: 10016
  Total removal areas: 1010
  Structures with multiple removals: 290 (29.0%)
Saved updated file (all structures, points kept) to:
c:\Users\villemev\Documents\VS Projects\DGCNN_Grid_and_Substructure_Classification\JSON\rotated_combination_arc_skew_ort_3-15_12-20_n1_more_random2_all_rn42_shifted_points.json
Loaded 1000 buildings


interactive(children=(IntSlider(value=0, description='idx', max=999), Output()), _dom_classes=('widget-interac…

<function Training_sett_scripts.plot_structure(idx, structures)>

## Clean Dataset
Remove ambiguous near-axis samples to produce a cleaner training JSON.

In [5]:
# Clean ambiguous training points near opposite axes and save a fixed JSON
from Data_cleaning_scripts import clean_training_json_axis_conflicts
import os

# Input JSON to clean
input_json = out_path  # or set a specific file path, e.g. "JSON/rotated_combination_arc_skew_ort_3-15_4-8_n1_shifted_points.json"

# Distance threshold in mm
threshold_mm = 1000

# Optional output path (if None -> appends "_axis_cleaned" before .json)
output_json = None

report = clean_training_json_axis_conflicts(
    input_json_path=input_json,
    threshold_mm=threshold_mm,
    output_json_path=output_json,
    skip_curved=True  # keeps fan/arc/curved structures unchanged

)

print("Saved fixed file:")
print(os.path.abspath(report['output_json_path']))

print("\nGlobal stats:")
for k, v in report['stats'].items():
    print(f"  {k}: {v}")

# Optional: inspect the first few structure-level rows
import pandas as pd
display(pd.DataFrame(report['per_structure_stats']).head(10))

Saved fixed file:
c:\Users\villemev\Documents\VS Projects\DGCNN_Grid_and_Substructure_Classification\JSON\rotated_combination_arc_skew_ort_3-15_12-20_n1_more_random2_all_rn42_shifted_points_axis_cleaned.json

Global stats:
  structures_total: 1000
  structures_cleaned: 677
  points_before: 143286
  points_after: 138459
  removed_total: 4827
  removed_outlier_near_axis: 2738
  removed_u_axis_near_v_axis: 995
  removed_v_axis_near_u_axis: 1094
  skipped_structures: 269


,structure,points_before,points_after,removed_total,removed_outlier_near_axis,removed_u_axis_near_v_axis,removed_v_axis_near_u_axis,skipped,skip_reason
0,0,136,129,7,1,3,3,False,NaN
1,1,118,118,0,0,0,0,True,curved structure_type='fan'
2,2,47,47,0,0,0,0,True,curved structure_type='fan'
3,3,179,167,12,4,3,5,False,NaN
4,4,308,294,14,9,1,4,False,NaN
5,5,213,202,11,8,2,1,False,NaN
6,6,51,48,3,3,0,0,False,NaN
7,7,33,33,0,0,0,0,False,NaN
8,8,156,151,5,5,0,0,False,NaN
9,9,53,53,0,0,0,0,True,curved structure_type='fan'


## Visualize Cleaned Dataset
Interactive preview of the cleaned structures for quality checks.

In [ ]:
# Visualize cleaned dataset
in_file = report["output_json_path"]
with open(in_file, "r") as f:
    data = json.load(f)

max_n = 1000
structures = data["structures"][:max_n]
print(f"Loaded {len(structures)} structures")

max_display = min(max_n, len(structures))
print(f"Displaying first {max_display} structures")

widgets.interact(
    plot_structure,
    idx=widgets.IntSlider(min=0, max=max_display - 1, step=1, value=0, description="Structure:"),
    structures=widgets.fixed(structures),
)

Loaded 1000 buildings


interactive(children=(IntSlider(value=0, description='idx', max=999), Output()), _dom_classes=('widget-interac…

<function Training_sett_scripts.plot_structure(idx, structures)>

## Dataset Sweep And Export
Use the cells below to generate multiple JSON training sets with controlled size and grid-range settings.

- Reproducibility: fixed random seed.
- Output folders: `JSON_sweep` and `JSON_sweep_grid`.
- Naming convention: includes grid range, sample count, and seed.

### Helper Functions
Defines reusable generators and JSON export helpers used by the sweep sections below.

In [7]:
import os
import json
import random
from pathlib import Path




def generate_mixed_structures(
    n_structures,
    min_grid_size=3,
    max_grid_size=15,
    seed=42,
    shifted_points=True,
    point_count_domain=(1, 5),
    dx_domain_mm=(-50.0, 50.0),
    dy_domain_mm=(-50.0, 50.0),
):
    """
    Generates a mixed dataset with:
      25% arc/curved
      25% skewed
      25% rotated with extra columns
      25% rotated with partial removal

    Returns:
        list of structures
    """

    rng = random.Random(seed)
    structures = []

    for i in range(n_structures):
        r_float = rng.random()

        if r_float < 0.25:
            s = build_structure_with_arc_curvature(
                rng,
                rows_range=(min_grid_size, max_grid_size),
                cols_range=(min_grid_size, max_grid_size),
                row_gap_domain_mm=(4000, 6000),
                col_gap_domain_mm=(4000, 6000),
                rotation_domain_deg=(-45, 45),
                arc_curvature_domain=(-0.00001, 0.00001),
                random_columns_range=(0, 2),
                remove_points_range=(0, 2),
            )

        elif r_float < 0.50:
            s = build_structure_with_skew(
                rng,
                rows_range=(min_grid_size, max_grid_size),
                cols_range=(min_grid_size, max_grid_size),
                row_gap_domain_mm=(4000, 8000),
                col_gap_domain_mm=(4000, 8000),
                rotation_domain_deg=(-45, 45),
                skew_domain_deg=(-15, 15),
                random_columns_range=(0, 2),
                remove_points_range=(0, 2),
            )

        elif r_float < 0.75:
            s = build_structure_with_rotation_exCol(
                rng,
                rows_range=(min_grid_size, max_grid_size),
                cols_range=(min_grid_size, max_grid_size),
                row_gap_domain_mm=(4000, 8000),
                col_gap_domain_mm=(4000, 8000),
                rotation_domain_deg=(-45, 45),
                random_columns_range=(0, 2),
                remove_points_range=(0, 1),
            )

        else:
            s = build_structure_with_rotation_exCol_removePart(
                rng,
                rows_range=(min_grid_size, max_grid_size),
                cols_range=(min_grid_size, max_grid_size),
                row_gap_domain_mm=(4000, 8000),
                col_gap_domain_mm=(4000, 8000),
                rotation_domain_deg=(-45, 45),
                random_columns_range=(0, 2),
                remove_points_range=(0, 0),
                removal_probability=0.7,
            )

        structures.append(s)

    if shifted_points:
        for s in structures:
            shift_random_points(
                s,
                rng,
                point_count_domain,
                dx_domain_mm,
                dy_domain_mm,
            )

    return structures

def save_json_dataset(
    structures,
    out_path,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(exist_ok=True)

    with open(out_path, "w") as f:
        json.dump({"structures": structures}, f, indent=2)

    total_random_cols = sum(s.get("num_random_columns", 0) for s in structures)
    total_grid_points = sum(s.get("R", 0) * s.get("C", 0) for s in structures)

    avg_random_cols = total_random_cols / len(structures) if structures else 0
    avg_percentage = (
        total_random_cols / total_grid_points * 100
        if total_grid_points > 0
        else 0
    )

    max_R = max(s.get("R", 0) for s in structures)
    max_C = max(s.get("C", 0) for s in structures)

    print(f"Saved: {out_path}")
    print(f"  Structures: {len(structures)}")
    print(f"  Max R: {max_R}, Max C: {max_C}")
    print(f"  Random columns avg: {avg_random_cols:.2f}")
    print(f"  Random columns percentage: {avg_percentage:.2f}%")

### Sweep A: Fixed Max Grid Size
Generate datasets for increasing structure count while keeping grid-size upper bound fixed.

In [8]:
n_structures_values = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
max_grid_size_values = [15]

# Make sure this folder exists
OUT_DIR = Path("JSON_sweep")
OUT_DIR.mkdir(exist_ok=True)

for n_structures in n_structures_values:
    for max_grid_size in max_grid_size_values:
        min_grid_size = 3
        seed = 42

        structures = generate_mixed_structures(
            n_structures=n_structures,
            min_grid_size=min_grid_size,
            max_grid_size=max_grid_size,
            seed=seed,
            shifted_points=True,
            point_count_domain=(1, 5),
            dx_domain_mm=(-50.0, 50.0),
            dy_domain_mm=(-50.0, 50.0),
        )

        out_file = (
            OUT_DIR
            / f"mixed_grid_{min_grid_size}-{max_grid_size}"
              f"_n{n_structures}"
              f"_shifted"
              f"_seed{seed}.json"
        )

        save_json_dataset(structures, out_file)

Saved: JSON_sweep\mixed_grid_3-15_n1000_shifted_seed42.json
  Structures: 1000
  Max R: 15, Max C: 15
  Random columns avg: 0.94
  Random columns percentage: 1.21%
Saved: JSON_sweep\mixed_grid_3-15_n2000_shifted_seed42.json
  Structures: 2000
  Max R: 15, Max C: 15
  Random columns avg: 0.97
  Random columns percentage: 1.22%
Saved: JSON_sweep\mixed_grid_3-15_n3000_shifted_seed42.json
  Structures: 3000
  Max R: 15, Max C: 15
  Random columns avg: 0.97
  Random columns percentage: 1.22%
Saved: JSON_sweep\mixed_grid_3-15_n4000_shifted_seed42.json
  Structures: 4000
  Max R: 15, Max C: 15
  Random columns avg: 0.97
  Random columns percentage: 1.21%
Saved: JSON_sweep\mixed_grid_3-15_n5000_shifted_seed42.json
  Structures: 5000
  Max R: 15, Max C: 15
  Random columns avg: 0.97
  Random columns percentage: 1.20%
Saved: JSON_sweep\mixed_grid_3-15_n6000_shifted_seed42.json
  Structures: 6000
  Max R: 15, Max C: 15
  Random columns avg: 0.96
  Random columns percentage: 1.19%
Saved: JSON_swee

### Sweep B: Fixed Structure Count
Generate datasets for multiple grid-size limits while keeping structure count fixed.

In [9]:
n_structures_values = [1000]
max_grid_size_values = [4,6,8,10,12,14,16,18,20]

# Make sure this folder exists
OUT_DIR = Path("JSON_sweep_grid")
OUT_DIR.mkdir(exist_ok=True)

for n_structures in n_structures_values:
    for max_grid_size in max_grid_size_values:
        min_grid_size = 3
        seed = 42

        structures = generate_mixed_structures(
            n_structures=n_structures,
            min_grid_size=min_grid_size,
            max_grid_size=max_grid_size,
            seed=seed,
            shifted_points=True,
            point_count_domain=(1, 5),
            dx_domain_mm=(-50.0, 50.0),
            dy_domain_mm=(-50.0, 50.0),
        )

        out_file = (
            OUT_DIR
            / f"mixed_grid_{min_grid_size}-{max_grid_size}"
              f"_n{n_structures}"
              f"_shifted"
              f"_seed{seed}.json"
        )

        save_json_dataset(structures, out_file)

Saved: JSON_sweep_grid\mixed_grid_3-4_n1000_shifted_seed42.json
  Structures: 1000
  Max R: 4, Max C: 4
  Random columns avg: 0.51
  Random columns percentage: 4.13%
Saved: JSON_sweep_grid\mixed_grid_3-6_n1000_shifted_seed42.json
  Structures: 1000
  Max R: 6, Max C: 6
  Random columns avg: 0.78
  Random columns percentage: 3.78%
Saved: JSON_sweep_grid\mixed_grid_3-8_n1000_shifted_seed42.json
  Structures: 1000
  Max R: 8, Max C: 8
  Random columns avg: 0.93
  Random columns percentage: 3.01%
Saved: JSON_sweep_grid\mixed_grid_3-10_n1000_shifted_seed42.json
  Structures: 1000
  Max R: 10, Max C: 10
  Random columns avg: 0.95
  Random columns percentage: 2.28%
Saved: JSON_sweep_grid\mixed_grid_3-12_n1000_shifted_seed42.json
  Structures: 1000
  Max R: 12, Max C: 12
  Random columns avg: 1.00
  Random columns percentage: 1.77%
Saved: JSON_sweep_grid\mixed_grid_3-14_n1000_shifted_seed42.json
  Structures: 1000
  Max R: 14, Max C: 14
  Random columns avg: 0.98
  Random columns percentage: 1